# NB 67 — Cross-method comparison by cosine(steered text, concept)

The directed-expression effect used so far is in each method's own units (frame
trace-correlation vs subspace ‖B·h‖, a ~250× scale gap), so it can't compare
**multi-FRH** against **subspace** directly. This notebook re-scores the *same*
steered continuations (reused from NB 65, `65_source_battery.jsonl` — **no new
generation**) with a bounded, scale-invariant metric:

> `token_cos(text, span) = mean_t ‖Fᵀ h_t‖ / ‖h_t‖ ∈ [0, 1]`

the fraction of the text's hidden-state norm lying in the concept span
(`frames.linalg.angles.vector_subspace_cosine`; harness metric-v3 `token_cos`).
It takes any orthonormal span, so it applies identically to a frame concept
(`concept.tensor`, k≤4 dims) and a subspace concept (`concept.basis`, r=8 dims).

**Two views, both cross-method comparable:**

- **Δcos (self-ref)** — each method's steered text measured against the span it
  steered toward (frame span for mfrh, subspace basis for subspace), minus the
  unsteered baseline. Δ controls for the span's dimensionality offset.
- **Δcos (frame-ref)** — *both* methods measured against the **same** frame
  concept span (k≤4 dims), fully removing the span-dimensionality confound: a
  common target for a clean head-to-head.

Both are sign-weighted over a config's parts (single `+ΔA`; blend `+ΔA+ΔB`;
repel `+ΔA−ΔB`), aggregated per method / source / case / λ.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.linalg.angles import vector_subspace_cosine
from frames.representations import FrameUnembeddingRepresentation
from subspace import SubspaceUnembeddingRepresentation

In [ ]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
CONCEPTS_JSON = "resources/64_custom_concepts.json"
WORD_FIELD = "core"
MAX_TOKENS, STEPS, RANK, MIN_LEMMAS_8 = 4, 24, 8, 2
SUFFIX = ""

SOURCES = ["core", "wn8lang"]
SYNSET_MAP = {"joy": "joy.n.01", "sorrow": "sorrow.n.01",
              "safe": "safe.a.01", "harm": "injury.n.01"}

SMOKE = True  # smoke -> reuse the NB65 smoke continuations; full -> the full battery

In [ ]:
SRC_JSONL = f"resources/65_source_battery{'_smoke' if SMOKE else ''}.jsonl"
if SMOKE:
    SUFFIX = SUFFIX or "_smoke"
SURFACE_VARIATIONS = ("{term}", " {term}", "{term} ", " {term} ")


def custom_concept_dataframe(tokenizer, name, words, max_token_count):
    pad_id, bos_id = tokenizer.pad_token_id, tokenizer.bos_token_id
    rows = []
    for w in words:
        for v in SURFACE_VARIATIONS:
            ids = tokenizer(v.format(term=w.strip()), add_special_tokens=False)["input_ids"]
            if ids and ids[0] == bos_id:
                ids = ids[1:]
            if 0 < len(ids) <= max_token_count:
                rows.append(ids + [pad_id] * (max_token_count - len(ids)))
    if not rows:
        raise ValueError(f"no variation of {name!r} fits max_token_count={max_token_count}")
    return pd.DataFrame({"synset": [name], "tokens": [np.array(rows, dtype=np.int64)]})


def parse_parts(kind, name):
    if kind == "single":
        return [(name, +1)]
    if kind == "blend":
        a, b = name.split("+"); return [(a, +1), (b, +1)]
    a, b = name.split("-"); return [(a, +1), (b, -1)]  # repel

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16,
)
sur = SubspaceUnembeddingRepresentation(
    model=fur.model, unembedding=fur, data=fur.data, rank=RANK, method="pooled",
)
harness = EvaluationHarness(fur, Path("resources/_nb67_scratch.jsonl"), MIN_LEMMAS_8, MAX_TOKENS)
harness.member_lemmas("joy.n.01")  # force WordNet parquet load out of loops

WORDS = {name: e[WORD_FIELD] for e in json.loads(Path(CONCEPTS_JSON).read_text())
         for name in [e["concept"]] if name in SYNSET_MAP}

In [ ]:
# Concept spans, keyed by (source, name). Frame span = tensor (d,k); subspace
# span = basis (d,r). Both orthonormal -> valid for vector_subspace_cosine.
frame_span, sub_span = {}, {}
tok = fur.model.tokenizer
for name, synset in SYNSET_MAP.items():
    df = custom_concept_dataframe(tok, name, WORDS[name], MAX_TOKENS)
    frame_span[("core", name)] = fur.compute_concept(df).tensor.squeeze(0)
    sub_span[("core", name)] = sur.compute_concept(df).basis
    frame_span[("wn8lang", name)] = fur.get_concept_cached(synset, MIN_LEMMAS_8, MAX_TOKENS).tensor.squeeze(0)
    sub_span[("wn8lang", name)] = sur.get_concept_cached(synset, MIN_LEMMAS_8, MAX_TOKENS, method="pooled").basis
print("spans built:", len(frame_span), "frame,", len(sub_span), "subspace")

In [ ]:
records = [json.loads(x) for x in Path(SRC_JSONL).read_text().strip().split("\n") if x]
print(f"{len(records)} logged generations from {SRC_JSONL}")

# Regenerate the unsteered baselines for the prompts in the log (greedy, cheap).
prompts = sorted({r["prompt"] for r in records})
baseline_texts = fur.generate(prompts, max_new_tokens=STEPS + 2, do_sample=False)
baseline_cont = {p: harness.continuation(t, p) for p, t in zip(prompts, baseline_texts)}
print(f"{len(prompts)} baseline continuations regenerated")

In [ ]:
hidden_cache = {}


def hidden_of(text):
    if text not in hidden_cache:
        with torch.inference_mode():
            h = fur.model.forward_last_hiden_state(fur.model.tokenize(text))
        hidden_cache[text] = h.squeeze(0)
    return hidden_cache[text]


def token_cos(text, span):
    if not text.strip():
        return 0.0
    h = hidden_of(text).to(span.device)
    return float(vector_subspace_cosine(h, span).mean())


rows = []
for r in records:
    c = r["config"]
    parts = parse_parts(c["kind"], c["name"])
    steered, base = r["continuation"], baseline_cont[r["prompt"]]
    dcos_self, dcos_frameref = 0.0, 0.0
    for cname, sign in parts:
        fspan = frame_span[(c["source"], cname)]
        own = fspan if c["method"] == "multifrh" else sub_span[(c["source"], cname)]
        dcos_self += sign * (token_cos(steered, own) - token_cos(base, own))
        dcos_frameref += sign * (token_cos(steered, fspan) - token_cos(base, fspan))
    rows.append({"method": c["method"], "source": c["source"], "case": c["case"],
                 "kind": c["kind"], "lam": c["lam"],
                 "dcos_self": dcos_self, "dcos_frameref": dcos_frameref})
rdf = pd.DataFrame(rows)
print(f"{len(rdf)} re-scored | unique texts embedded: {len(hidden_cache)}")

In [ ]:
summary = (
    rdf.groupby(["method", "source", "case", "lam"])
    .agg(dcos_self=("dcos_self", "mean"), dcos_frameref=("dcos_frameref", "mean"),
         n=("dcos_self", "size"))
    .reset_index()
)
summary.to_csv(f"resources/67_cosine_summary{SUFFIX}.csv", index=False)
print("=== Δcos (self-ref) and Δcos (frame-ref), per method/source/case/λ ===")
print(summary.round(4).to_string(index=False))

# Head-to-head: multifrh − subspace on the common frame-ref cosine (fully comparable).
print("\n=== head-to-head Δcos (frame-ref): multifrh − subspace ===")
h = summary.pivot_table(index=["source", "case", "lam"], columns="method", values="dcos_frameref")
h["mfrh−sub"] = h["multifrh"] - h["subspace"]
print(h.round(4).to_string())

In [ ]:
LAMS = sorted(rdf.lam.unique())
CASES = ["1-concept", "2-concept"]
mcolors = {"multifrh": "tab:green", "subspace": "tab:blue"}
metrics = [("dcos_frameref", "Δcos vs frame span (common target)"),
           ("dcos_self", "Δcos vs own span (self-ref)")]
x = np.arange(len(CASES)); width = 0.38
fig, axes = plt.subplots(len(SOURCES), len(metrics), figsize=(5 * len(metrics), 3.4 * len(SOURCES)), squeeze=False)
lam_plot = LAMS[-1]
for row, src in enumerate(SOURCES):
    for col, (metric, label) in enumerate(metrics):
        ax = axes[row][col]
        for j, m in enumerate(["multifrh", "subspace"]):
            vals = [summary[(summary.method == m) & (summary.source == src)
                            & (summary.case == c) & (summary.lam == lam_plot)][metric].mean()
                    for c in CASES]
            ax.bar(x + (j - 0.5) * width, vals, width, label=m, color=mcolors[m])
        ax.set_xticks(x); ax.set_xticklabels(CASES)
        ax.set_title(f"{src} — {label}")
        ax.axhline(0, color="gray", lw=0.6); ax.legend(fontsize=8)
plt.suptitle(f"Cosine steering gain — multi-FRH vs subspace (λ={lam_plot}, bounded [0,1])")
plt.tight_layout()
plt.savefig(f"resources/67_cosine_cross_method{SUFFIX}.png", dpi=150)
plt.show()

## Notes

- **Δcos is bounded and scale-invariant**, so unlike the directed-expression
  effect it is directly comparable across methods. `dcos_frameref` (both methods
  vs the *same* frame span) is the cleanest head-to-head; `dcos_self` reports
  each method's gain into the span it actually steered toward.
- Reuses `65_source_battery.jsonl` continuations; only re-embeds text and
  regenerates the greedy baselines. Flip `SMOKE = False` for the full battery.
- Positive Δcos = steering moved the text's hidden states into the concept span
  (relative to the unsteered baseline).